# Atlas samples: 2025 VCC, Tahoe-100M, scBaseCount

Pulls a small, fast-loading sample of each of the three datasets so we can work out what
cleaning and combining is needed before committing to compute.

Access is anonymous and free (public bucket, no GCP account). Full sizes and the commands to
pull everything are in the last section.

Helpers live in `src/atlas.py`.

In [1]:
import sys
from pathlib import Path

# Find the repo root by walking up to pyproject.toml. Don't rely on the kernel's cwd --
# VS Code and jupyter lab disagree about whether it's the notebook dir or the repo root.
ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

try:
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp

    import atlas
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"{e.name!r} is missing, so this kernel is NOT the project .venv. "
        f"Running: {sys.executable} -- expected: {ROOT / '.venv' / 'Scripts' / 'python.exe'}. "
        'Switch kernel to "VCC virtual_cell_notebooks (.venv)" and re-run. '
        'Beware a kernel named only "Python (.venv)" -- it may belong to another repo.'
    ) from e

DATA_2026 = ROOT / 'data' / 'validation_2026'
TAHOE_PLATE = 'h5ad/plate10_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad'
SCBASE_DIR = 'h5ad/GeneFull_Ex50pAS/Homo_sapiens'
N = 2000  # cells per sample

repo root: C:\Users\tzs7ch\Desktop\VCC\virtual_cell_notebooks


## 1. What's in each dataset

Metadata only. No file contents transferred.

In [2]:
info = {
    'vcc2025': atlas.peek('vcc2025', 'train/adata_Training.h5ad'),
    'tahoe100M': atlas.peek('tahoe100M', TAHOE_PLATE),
}
for k, v in info.items():
    print(k)
    for kk, vv in v.items():
        print(f'   {kk:10s} {vv}')

vcc2025
   cells      221273
   genes      18080
   nnz        1932554688
   density    0.483
   format     csr_matrix
   dtype      float32
   obs_cols   ['target_gene', 'guide_id', 'batch']
   var_index  _index
   var_cols   ['_index', 'gene_id']
   layers     []
tahoe100M
   cells      8044908
   genes      62710
   nnz        11507888845
   density    0.023
   format     csr_matrix
   dtype      float32
   obs_cols   ['sample', 'gene_count', 'tscp_count', 'mread_count', 'drugname_drugconc', 'drug', 'cell_line', 'sublibrary', 'BARCODE', 'pcnt_mito', 'S_score', 'G2M_score', 'phase', 'pass_filter', 'cell_name', 'plate']
   var_index  gene_name
   var_cols   ['gene_name']
   layers     []


## 2. Sample: 2025 VCC (CRISPRi knockdown, one cell line)

CSR, so we can slice rows straight out of the 15 GB file.

In [3]:
vcc = atlas.slice_cells('vcc2025', 'train/adata_Training.h5ad', 0, N)
print(vcc)
print(vcc.obs['target_gene'].value_counts().head())

AnnData object with n_obs × n_vars = 2000 × 18080
    obs: 'target_gene', 'guide_id', 'batch'
    var: 'gene_id'
target_gene
non-targeting    358
TMSB4X            45
HIRA              42
TADA1             41
PRCP              33
Name: count, dtype: int64


## 3. Sample: Tahoe-100M (drug perturbation, many cell lines)

Also CSR. Slicing 2000 cells out of a 26 GB plate file.

In [4]:
tahoe = atlas.slice_cells('tahoe100M', TAHOE_PLATE, 0, N)
print(tahoe)
print(tahoe.obs[['cell_line', 'drug', 'drugname_drugconc']].head())
print()
print('cell lines in this slice:', tahoe.obs['cell_line'].nunique())
print('drugs in this slice:', tahoe.obs['drug'].nunique())

AnnData object with n_obs × n_vars = 2000 × 62710
    obs: 'sample', 'gene_count', 'tscp_count', 'mread_count', 'drugname_drugconc', 'drug', 'cell_line', 'sublibrary', 'BARCODE', 'pcnt_mito', 'S_score', 'G2M_score', 'phase', 'pass_filter', 'cell_name', 'plate'
                     cell_line                      drug  \
01_001_001-lib_1681  CVCL_1478  Bestatin (hydrochloride)   
01_002_149-lib_1681  CVCL_0459  Bestatin (hydrochloride)   
01_003_052-lib_1681  CVCL_C466  Bestatin (hydrochloride)   
01_003_090-lib_1681  CVCL_1724  Bestatin (hydrochloride)   
01_003_093-lib_1681  CVCL_1285  Bestatin (hydrochloride)   

                                              drugname_drugconc  
01_001_001-lib_1681  [('Bestatin (hydrochloride)', 0.05, 'uM')]  
01_002_149-lib_1681  [('Bestatin (hydrochloride)', 0.05, 'uM')]  
01_003_052-lib_1681  [('Bestatin (hydrochloride)', 0.05, 'uM')]  
01_003_090-lib_1681  [('Bestatin (hydrochloride)', 0.05, 'uM')]  
01_003_093-lib_1681  [('Bestatin (hydrochloride)

## 4. Sample: scBaseCount (public scRNA-seq, no perturbation)

These files are **CSC**, so row slicing would touch every column block. They're also one file
per SRA accession and small, so we download a whole one instead.

Pick a *median-sized* accession, not the smallest: the smallest files are degenerate runs
(the very smallest is 47 cells with a median of 1 UMI, i.e. useless).

In [5]:
files = [f for f in atlas.ls('scbasecount', SCBASE_DIR, detail=True) if f['type'] != 'directory']
files.sort(key=lambda f: f.get('size') or 0)
print(f'{len(files)} accessions, sizes {files[0]["size"]/1e6:.1f} - {files[-1]["size"]/1e6:.1f} MB')

mid = files[len(files) // 2]
acc = mid['name'].rsplit('/', 1)[-1]
print(f'using median-sized: {acc} ({mid["size"] / 1e6:.1f} MB)')
path = atlas.fetch('scbasecount', f'{SCBASE_DIR}/{acc}')  # cached; re-runs don't re-download

import anndata as ad

scbase = ad.read_h5ad(path)
print(scbase)
print(scbase.obs[['SRX_accession', 'cell_type']].head())

35263 accessions, sizes 2.5 - 1087.5 MB
using median-sized: SRX19810932.h5ad (70.2 MB)


AnnData object with n_obs × n_vars = 4161 × 36601
    obs: 'gene_count_Unique', 'umi_count_Unique', 'gene_count_UniqueAndMult-EM', 'umi_count_UniqueAndMult-EM', 'gene_count_UniqueAndMult-Uniform', 'umi_count_UniqueAndMult-Uniform', 'SRX_accession', 'cell_type', 'cell_ontology_term_id'
    var: 'gene_symbols'
    layers: 'UniqueAndMult-EM', 'UniqueAndMult-Uniform'
                 SRX_accession cell_type
AAACCTGAGTCATCCA   SRX19810932          
AAACCTGAGTGCTGCC   SRX19810932          
AAACCTGCAGCCACCA   SRX19810932          
AAACCTGCATGATCCA   SRX19810932          
AAACCTGCATGCCTAA   SRX19810932          


## 5. Differences that need handling

The three don't line up. This is the actual work.

In [6]:
def summarize(name, a, gene_index_kind, pert_col, ctx_col, modality):
    X = a.X.tocsr() if sp.issparse(a.X) and a.X.format == 'csc' else a.X
    return {
        'dataset': name,
        'cells': a.n_obs,
        'genes': a.n_vars,
        'density': round(X.nnz / (a.n_obs * a.n_vars), 3),
        'median_umi': int(np.median(np.asarray(X.sum(axis=1)).ravel())),
        'gene_ids': gene_index_kind,
        'pert_col': pert_col,
        'context_col': ctx_col,
        'modality': modality,
    }


rows = [
    summarize('vcc2025', vcc, 'symbol (Ensembl in var.gene_id)', 'target_gene', 'none (H1 only)', 'CRISPRi knockdown'),
    summarize('tahoe100M', tahoe, 'symbol', 'drug', 'cell_line', 'drug'),
    summarize('scbasecount', scbase, 'Ensembl (symbol in var.gene_symbols)', 'none', 'cell_type', 'observational'),
]
pd.set_option('display.width', 200)
print(pd.DataFrame(rows).to_string(index=False))

    dataset  cells  genes  density  median_umi                             gene_ids    pert_col    context_col          modality
    vcc2025   2000  18080    0.462       44928      symbol (Ensembl in var.gene_id) target_gene none (H1 only) CRISPRi knockdown
  tahoe100M   2000  62710    0.020        1527                               symbol        drug      cell_line              drug
scbasecount   4161  36601    0.064        6026 Ensembl (symbol in var.gene_symbols)        none      cell_type     observational


### Notes

- **Sequencing depth is the biggest problem.** median UMI/cell is ~45k in 2025 vs ~1.5k in Tahoe
  — roughly 30x. Raw counts are not comparable across datasets; normalize per cell before
  combining anything.
- **Density follows from that**: 2025 is ~46% nonzero, Tahoe ~2%.
- **Gene spaces all differ**: 18,080 / 62,710 / 36,601, and 2026 is 18,533. None is a subset of
  another by position.
- **Two ID types**: 2025 and Tahoe index by symbol, scBaseCount by Ensembl. Both carry the other
  form in a `var` column, so map through Ensembl and merge on that.
- **Order differs even where symbols match.** Align by name, never by column position.
- **Matrix format differs**: CSR for 2025/Tahoe, CSC for scBaseCount.
- **Only 2025 has CRISPRi labels.** Tahoe gives cell-line diversity with the wrong perturbation
  type; scBaseCount gives cell-type diversity with no perturbation at all.
- **scBaseCount annotation is patchy** — `cell_type` is blank for some accessions, so it can't be
  assumed present.

## 6. Gene ID overlap

How much survives a merge, using our 2026 gene list as the target space.

In [7]:
genes_2026 = set(pd.read_csv(DATA_2026 / 'gene_names.csv').iloc[:, 0].astype(str))

sym = {
    'vcc2025': set(vcc.var.index.astype(str)),
    'tahoe100M': set(tahoe.var.index.astype(str)),
    'scbasecount': set(scbase.var['gene_symbols'].astype(str)),
}
for k, v in sym.items():
    print(f'{k:12s} symbols={len(v):6d}  overlap with 2026={len(v & genes_2026):6d}  '
          f'missing from it={len(genes_2026 - v):5d}')

print()
common = set.intersection(*sym.values()) & genes_2026
print(f'genes present in ALL THREE and in the 2026 list: {len(common)}')

vcc2025      symbols= 18080  overlap with 2026= 18077  missing from it=  456
tahoe100M    symbols= 62710  overlap with 2026= 18150  missing from it=  383
scbasecount  symbols= 36591  overlap with 2026= 18533  missing from it=    0

genes present in ALL THREE and in the 2026 list: 17739


## 7. Full data — template for when we have compute

Everything is free to download; the limit is disk and time (~35 MB/s measured).

| dataset | full size | notes |
|---|---|---|
| vcc2025 | 22.4 GB | train 15.5 + validation 6.9; the only CRISPRi ground truth |
| tahoe100M | 336.7 GB | 14 plate files, 23-37 GB each; `metadata/obs_metadata.parquet` is 2.3 GB |
| scbasecount | ~3,443 GB | human, one feature type: 35,263 accession files; per-organism metadata ~6 GB |

Downloads cache to `data/atlas_cache/` (gitignored). A cached file is not re-fetched; pass
`verify=False` to skip even the remote size check.

Don't pull scBaseCount wholesale — filter its `sample_metadata.parquet` first, then fetch only
the accessions you want.

In [8]:
# Set to True when there's disk and time. dry_run first prints sizes and checks headroom.
PULL_FULL = False
DRY_RUN = True

FULL = {
    'vcc2025': ['train/adata_Training.h5ad', 'validation/adata_Validation.h5ad'],
    'tahoe100M': ['metadata/obs_metadata.parquet', TAHOE_PLATE],
    # scbasecount: select accessions from sample_metadata.parquet, don't take all 35k
}

if PULL_FULL:
    for ds, paths in FULL.items():
        for sp_ in paths:
            if DRY_RUN:
                n = int(atlas.fs().info(atlas.remote(ds, sp_))['size'])
                cached = atlas.local_path(ds, sp_).exists()
                print(f'{ds:11s} {sp_:55s} {n / 1e9:8.2f} GB {"[cached]" if cached else ""}')
            else:
                print(atlas.fetch(ds, sp_))
else:
    total = 0
    for ds, paths in FULL.items():
        for sp_ in paths:
            total += int(atlas.fs().info(atlas.remote(ds, sp_))['size'])
    print(f'PULL_FULL is False. This selection would be {total / 1e9:.1f} GB.')
    print('Equivalent CLI:  python src/atlas.py get vcc2025 train/adata_Training.h5ad --dry-run')

PULL_FULL is False. This selection would be 51.2 GB.
Equivalent CLI:  python src/atlas.py get vcc2025 train/adata_Training.h5ad --dry-run
